# 02 - Pretraitement et histogramme

**Objectif de l'etape.** Comparer la conversion en niveaux de gris, le filtrage gaussien et les ameliorations de contraste par histogramme.

**Methode.** Le grayscale travaille sur la luminance, le filtre gaussien reduit le bruit, le stretching et l'egalisation modifient la dynamique des intensites.

**Lien avec le sujet.** Un pretraitement stable facilite la segmentation et le suivi Lucas-Kanade.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'data' / 'car' / 'car-11' / 'img'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

image_files = sorted([p for p in DATASET_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
print('Nombre de frames:', len(image_files))

Nombre de frames: 1661


In [2]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing import compare_preprocessing_methods, compute_histogram
from src.visualization import save_comparison_grid

frame = cv2.imread(str(image_files[0]))
methods = compare_preprocessing_methods(frame)
images = {'original': frame}
images.update(methods)
output_path = RESULTS_DIR / 'preprocessing' / 'preprocessing_histogram_comparison.png'
save_comparison_grid(images, output_path, title='Pretraitement et histogramme')
print('Image sauvegardee:', output_path)

Image sauvegardee: c:\Users\espacegamers\Desktop\Master IAII\Cours\drive-download-20251005T182208Z-1-001\S2\Traitement d_images et vision par ordinateur_\Projet\motion-estimation-project\results\preprocessing\preprocessing_histogram_comparison.png


In [3]:
plt.figure(figsize=(14, 8))
for i, (name, img) in enumerate(methods.items(), start=1):
    plt.subplot(2, 3, i)
    plt.imshow(img, cmap='gray')
    plt.title(name)
    plt.axis('off')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
for name in ['gray', 'stretching', 'equalization']:
    plt.plot(compute_histogram(methods[name]), label=name)
plt.title('Histogrammes')
plt.xlabel('Intensite')
plt.ylabel('Nombre de pixels')
plt.legend()
plt.grid(True)
plt.show()

C:\Users\espacegamers\AppData\Local\Temp\ipykernel_14032\3787815228.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\espacegamers\AppData\Local\Temp\ipykernel_14032\3787815228.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Resultats generes.** `results/preprocessing/preprocessing_histogram_comparison.png` compare l'image originale, grayscale, Gaussian, CLAHE, stretching et equalization.

**Interpretation.** Le stretching etend mieux les intensites sur `[0,255]`, ce qui aide a distinguer la voiture de la route. L'egalisation peut produire un contraste plus fort, mais elle peut aussi modifier agressivement les intensites et perturber l'hypothese d'illumination constante du flot optique. Pour la suite, `stretching` est un choix prudent.